<img src="images/corndel_logo.png" height="200">

# Core Python & pandas for data engineering




## Key

> 💻 Activity for you to do <br>

>❓ A question for you <br>

> ℹ️ Important note (e.g. data security reminder) <br>

> 📗 A useful guide or reference to review if time allows <br>

> 🚀 A going further stretch activity only if your time allows


## Background

In this notebook we introduce Python and pandas through a realistic business scenario.

Here are the learning outcomes for this notebook.

---

### Learning outcomes

1. Learn how to run Python in a browser-based development environment (JupyterLite).
2. Learn core Python and Pandas skills for data manipulation and analysis within a data engineering scenario.
3. Develop key data manipulation skills with Pandas DataFrames, including selection, filtering, and creating new columns.
4. Manipulate date and time data with Pandas.
5. Begin data quality checks in Python using automated Exploratory Data Analysis (EDA)

---

### Running this notebook in your browser

This notebook runs in JupyterLite: Jupyter and Python running inside your web browser. There is nothing to install, and the data files the notebook needs are already in this lab, alongside the notebook.

A few things work differently from a Python installation on your own machine:

- Packages that are not built in are installed with a `%pip install` cell just before they are first used. Run those cells when you reach them.
- Your changes are saved in your browser's storage. To keep a copy elsewhere, right-click the notebook in the file browser on the left and select **Download**.
- Files your code writes, such as CSV or Parquet files, appear in the file browser on the left.

> **ℹ️ Do not** upload your organisational data here. Use this notebook only with the artificial data provided. Any use of organisational data must be evaluated, approved, and documented within your organisation's policies and procedures.

> 📗 The original version of this notebook was run in GitHub Codespaces, a cloud-based VS Code environment. The Aptem module "2.1 Core SQL and Python for data engineering" includes a video of that set-up. You do not need Codespaces to complete the exercises here.

---

### Teaching through examples

The teaching approach in this notebook follows David Robinson's motto, *"Don't teach student's the hard way first"* : http://varianceexplained.org/r/teach-hard-way/ And is expanded upon in this quote:

> ***Get students doing powerful things quickly:*** *Most of my educational philosophy can be boiled down to this statement: have goals for what your students should be able to do, and start them doing it as soon as possible.* http://varianceexplained.org/r/teach-tidyverse/

So in this spirit, of doing powerful things quickly, our goal is to extract and transform a dataset using Python ready for a customer. 

In doing this we will learn many core Python and pandas topics, in the context of a realistic business scenario.

So let's start with that business scenario.


---

### NHS Data Engineering Task

As a data engineer in the NHS, you're collaborating with a data analyst to build a BI dashboard using new Accident & Emergency data. The analyst, learning from past challenges, recognises the value of engineering the data for robustness and longevity of the dashboard.

<img src="images/scenario.png" width="400" />

Your role involves initial data exploration and transformation. This step is crucial for understanding data structure, quality, and potential insights. The data, while anonymised, could indirectly identify individuals, posing privacy risks.

To mitigate this, you'll work with artificial data mirroring the NHS A&E dataset. This is standard practice to create a development environment that reflects the production environment but is more manageable and secure. The artificial data, used for developing and refining code, ensures safe, accurate handling when applied to real data.

The complete dataset has a million rows. For efficiency, the analyst provides you wtih a 10,000-row sample to avoid long processing times.

These two features of a smaller and non-sensitive data set are common approaches to create what's called a [development environment](https://github.com/Corndel/DE-glossary?tab=readme-ov-file#development-environment) that mirrors the [production environment](https://github.com/Corndel/DE-glossary?tab=readme-ov-file#production-environment) (i.e. the full sized and unmasked data) but with manageable data sizes and controlled conditions.

Your tasks include:

1. Carrying out exploratory data analysis (EDA) of the A&E data to understand its structure and quality.
2. Agree and select the essential columns required for the metrics and visualisations of the BI dashboard.
3. Assess the data for missing or unusual values to ensure its accuracy and reliability.
4. Agree and carry out the necessary data transformations for the dashboard.

> 📗 The artificial data you will use is a synthetic dataset mimicking the structure of real NHS A&E data, enabling safe development and testing and was indeed created *"for building and testing data pipelines"*:
https://digital.nhs.uk/services/artificial-data#access-artificial-data-sets

## Python and pandas exercises by example

Let's now go through pandas and python exercises in a logical order to address the business scenario for the data analyst starting with reviewing any documentation.

---

### 0 Review data models or data dictionaries

Before exploring the data itself, you discussed with the Data Analyst if there was any official documentation, such as data models and dictionaries you could review.

It's important to seek such documentation to understand data structure, constraints, and definitions that apply to your data as it can significantly improve the subsequent stages of data analysis and engineering.

The analyst points you to three places where this information can be found:

- [NHS Data Dictionary Main Page](https://www.datadictionary.nhs.uk/index.html)
- [Hospital Episode Statistics Support Information](https://www.datadictionary.nhs.uk/supporting_information/hospital_episode_statistics.html)
- [Hospital Episode Statistics Data Dictionary Service](https://digital.nhs.uk/data-and-information/data-tools-and-services/data-services/hospital-episode-statistics/hospital-episode-statistics-data-dictionary)

In the last link above, you find the data dictionary. It has also been downloaded to this lab and is the Excel file `HES+TOS+V1.17.xlsx`. The information is rich and can be used in many ways, for example:

- **Validation**: Make sure data is correct and follows rules (e.g., 'Age at Activity Date' should be 0 or more and not a random symbol or letter).
- **Schema Design**: When eventually loaded to a data warehouse, you could create a database structure that fits the data (e.g., store 'Department Type' as CHAR(2) because it's always 2 characters long).
- **Anonymisation**: Remove or hide data that could identify a person (e.g., don't include 'Attendance Number' in any public dataset).
- **Documentation**: Write explanations of the data so anyone can understand it (e.g., explain that 'Department Type' '01' means a 24-hour emergency department).
- **Integration**: Integrate fields like 'AT_GP_PRACTICE' (Area Team of GP Practice) with external geographic information systems to analyse healthcare service usage patterns across different regions.
- **Enrichment**: Add extra information to the data to make it more meaningful (e.g., describe what each 'Incident Location Type' code means, like '10' for home or '50' for school).

> 💻 In the Excel data dictionary `HES+TOS+V1.17.xlsx`, quickly browse the `Known Data Item Issues` sheet. You can download the file from the file browser on the left and open it in Excel, or run the two code cells below to view the sheet here.

> ❓ How might one of these known data issues impact how you explore and test the quality of a particular data item?

In [ ]:
# openpyxl lets pandas read Excel (.xlsx) files in the browser
%pip install -q openpyxl

In [ ]:
import pandas as pd

# Read the 'Known Data Item Issues' sheet. header=1 uses the sheet's second row as the column names.
known_issues = pd.read_excel('HES+TOS+V1.17.xlsx', sheet_name='Known Data Item Issues', header=1)
known_issues

---

### 1 Import data and check data types

![Pandas Read/Write](images/02_io_readwrite.svg)

*Image from: https://pandas.pydata.org/docs/getting_started/intro_tutorials/02_read_write.html (pandas documentation, BSD 3-Clause licence)*

First we use the Python package `pandas` to import the artificial Accident & Emergency data set into a DataFrame and then inspect the data types of each column.

Correct data types are essential for efficient data processing and analysis. For instance, if a date is held as a string, it is more difficult (and inefficient) to identify and extract parts of the date or time from a string than the correct date format.

We'll also perform a preliminary assessment of the data to identify any immediate inconsistencies like missing values that could impact our subsequent data transformation and analysis steps.

In the code below when we import pandas (`import pandas as pd`), this is bringing into our Python runtime environment a powerful, flexible data analysis and manipulation library specifically designed for Python.

Think of pandas like Excel but in Python; it contains data structures and functions that make it easy to work with structured data. The primary data structure in pandas is the DataFrame, which you can think of as like an Excel sheet of data. In a DataFrame, our data is organised into rows and columns, and each column can be of a different type (like strings, integers, or dates), just like in an Excel sheet.

Pandas is built on Python code, among other languages like C and Cython. This means we are freed from coding in pure Python and the inherent limitations that come with it for data analysis and exploration. Instead, we get the ease of Python with the performance benefits of more [low-level languages](https://github.com/Corndel/DE-glossary?tab=readme-ov-file#low-level-language) like C. 

> 📗 With pandas, tasks like reading a CSV file, filtering data, grouping data, and performing calculations become much more straightforward and efficient. Competitor packages to pandas can offer even faster performance when processing data, such as [polars](https://docs.pola.rs/).

In [ ]:
# First, let's load the provided CSV file to examine its contents and structure.
import pandas as pd

# Set option to display all columns
pd.set_option('display.max_columns', None)

In Python, everything from variables, functions to classes is an object.

In the code below, `pd.read_csv()` is a function in the pandas library - represented by `pd`. It reads the CSV file whose location is stored in `file_path`. Here that is a file in this lab, next to the notebook.

The result, will be stored in `df` that is not just a variable - but is an object, specifically a pandas DataFrame object, a data structure allowing for easy data manipulation and analysis.

This shows how Python uses an [object-oriented](https://github.com/Corndel/DE-glossary?tab=readme-ov-file#object-oriented-programming) approach where classes act as templates for creating objects.

In [ ]:
# Load the CSV file
# (originally read from https://raw.githubusercontent.com/Corndel/corndel-datasets/main/nhs/artificial_hes_ae_2122.csv)
file_path = 'artificial_hes_ae_2122.csv'
df = pd.read_csv(file_path)

---

### 2 The pandas DataFrame

![Pandas data table](images/01_table_dataframe.svg)

*Image from: https://pandas.pydata.org/docs/getting_started/intro_tutorials/01_table_oriented.html (pandas documentation, BSD 3-Clause licence)*

The code below of `df.head(n=5)` is used to display the first few rows of the DataFrame data. This action is again a good example of Python's object-oriented nature. 

Here, `df`, which is an object created (or instantiated) from the pandas DataFrame class, is using its method `.head(n=5)` to show the first 5 rows of the dataset. 

The `.head()` method is a built-in function of DataFrame objects in pandas, allowing for easy inspection of data. We know it is a method (or function) because we call it with brackets into which we can pass arguments. In this case the `.head()` method only has one argument `n` which is the number of rows to display.

> 📗 W3 Schools provides a clear example of this concept by showing a class object (about a person) with attributes (name and age), and a function is included in the class as a method (`myfunc` that prints their name and age). Then, when the `Person` class is instantiated as a new object (`p1`), the attributes are initialised, and then the method (or function bound to that object) is run, using the assigned attributes: https://www.w3schools.com/python/gloss_python_object_methods.asp Creating a new DataFrame in pandas is fundamentally the same, it's just that the DataFrame class has many more attributes and methods geared towards data analysis. We will actually look at all of them shortly!

In [ ]:
# Display the first 5 rows of the dataframe to understand its structure and contents
df.head(n=5)

> 📗 How do we know what arguments we can pass to a pandas DataFrame method? Good programmers and data engineers become comfortable with visiting and using documentation.  We can look up all the attributes and methods bound to a pandas DataFrame here: https://pandas.pydata.org/docs/reference/frame.html And here is the documentation for the `.head()` method: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html

> 💻 Read the Pandas documentation for the `.tail()` method and use it to view the last 5 rows of the data frame in the code cell below: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.tail.html

In [ ]:
# Use the tail method on the DataFrame df


---

### 3 Attributes (characteristics) of a pandas DataFrame

The code below, `data.columns`, displays the column names of the DataFrame `df`. 

`.columns` is a DataFrame *attribute* that is one way to quickly see what columns are present in our dataset.

Notice that pandas DataFrame attributes like `.columns` have no brackets? This is because an attribute is **not** a method (a function bound to the object) so we do not (and cannot) pass any arguments to it like we did with the `.head()` method above.

In [ ]:
print(df.columns)

> 💻 Another useful attribute of a pandas DataFrame is `.shape`. Try running the `.shape` attribute below on the `df` DataFrame. 

In [ ]:
# Add the .shape attribute to the dataframe .df



> ❓ What does the output for `.shape` show you? Read the documentation https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html 

---

### 4 Methods (built in functions) and attributes (characteristics) of a pandas DataFrame 

Not only does the Pandas DataFrame object have `.head()` and `.columns`, but many more built-in attributes and methods that can significantly enhance and speed up data manipulation and analysis.

As well as the API documentation for a pandas [DataFrame](https://pandas.pydata.org/docs/reference/frame.html), another way to explore these functionalities is to use Python `dir()` function.

`dir()` lists all the attributes and methods of an object in Python, such as Pandas DataFrames. This is particularly useful in pandas as it uncovers all the available methods and attributes of the DataFrame object that could lead to more efficient and effective code development. For example, if you discovered a built-in method and used that instead of writing custom code. 

For instance, instead of manually iterating over rows to calculate statistics, you might find a DataFrame method like `.describe()` does exactly what you need in one line of code! Magic 🪄

In [ ]:
all_attributes = dir(df)
all_attributes

> 💻 Start by running the `.describe()` method on the `df` DataFrame without any parameters. Notice which columns are included in the summary and what kind of statistics it provides.
 
> 💻 Next, try running `.describe(include='all')` to see how the output changes when you include all columns. Finally, explore `.describe(include=[object])` to view the summary statistics for only non-numeric columns.
 
> Check out the [documentation for `.describe()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html). Reviewing the examples at the bottom can show you different ways to use the `include` and `exclude` parameters to control which types of data are summarised. This is often a helpful way to start with new documentation—examples can illustrate practical ways to use different options and variations.

In [ ]:
# Use the .describe() method on the DataFrame df


---

### 5 Getting to know your variables with the `%whos` Command

Before proceeding with further analysis, let's explore the `%whos` magic command.

Unlike in Excel where we can more easily see our data and what we have done to it by looking at the Excel sheets and formulas, in Python the equivalent objects are not so visible. This is where `%whos` can help us.

`%whos` provides a concise overview of all the variables (or objects) in your Jupyter Notebook's global [namespace](https://realpython.com/python-namespaces-scope/#namespaces-in-python). It's particularly useful for summarising the existing variables, their types, and key details.

When you run `%whos`, the output includes:

- `Variable`: The names of all variables in the global namespace.
- `Type`: The data type or class of each variable.
- `Data/Info`: A brief summary or key information about the variables.

As shown by the `%whos` output below, there are several objects in your global namespace so far that include:

1. `all_attributes` : A list of the methods and attributes in a pandas DataFrame.
1. `df`: A DataFrame object containing approximately 10,000 rows and 165 columns.
2. `file_path`: A string variable that stores the location of your CSV file.
3. `pd`: An alias for the pandas library, loaded as a module object.

In [ ]:
%whos

Another way to inspect the type of our objects one-by-one is with the Python `type()` function.

By knowing the type of an object we can work out what operations can be performed on the object and it can help you debug. 

> 💻 For example, running the three cells below, `type(pd)` tells you that `pd` is a module (pandas library), `type(df)` indicates `df` is a DataFrame, and `type(file_path)` confirms `file_path` is a string. While for debugging, often errors returned are from applying a function or operation that is not appropriate to the object type (e.g. applying arithmetic to a string).

In [ ]:
type(pd)

In [ ]:
type(df)

In [ ]:
type(file_path)

> 📗 So far we have used two built-in Python functions, `dir()` and `type()`. Here's all of the built-in Python functions in alphabetical order from the official Python documentation: https://docs.python.org/3/library/functions.html

> 💻 Use the following Python Built-in Functions len(), max(), min(), id() on your DataFrame. Read what the function does in the [documentation](https://docs.python.org/3/library/functions.html) 

In [ ]:
# Use the python built-in functions len(), max(), min(), id() on your DataFrame DF, what does the result tell you about the DataFrame object


> ❓ Explain to yourself what each function is telling you about DataFrame `df` and why each function could be useful when assessing a new data set

---

### 6 Using Python lists to store information

Our customer, the data analyst, is only interested in a subset of fields to go into their dashboard. 

Let's create a Python list of only the columns they require. By enclosing the comma-separated list in square brackets, we are telling Python that we want the object to be a list.

The `columns_to_prepare` list below includes the names of the columns we need. Lists in Python are mutable (they can be changed) and ordered (they have an order).

> 📗 W3 Schools Python lists: https://www.w3schools.com/python/python_lists.asp

In [ ]:
# A list for selecting the columns we want later
columns_to_prepare = [
    'FYEAR',
    'PARTYEAR',
    'AEKEY',
    'ARRIVALAGE',	
    'ARRIVALDATE',
    'AEARRIVALMODE',
    'ARRIVALTIME',
    'CONCLDUR',
    'CONCLTIME',
    'DEPDUR',
    'DEPTIME',
    'INITDUR',
    'INITTIME',
    'TRETDUR',
    'TRETTIME',
    'PROVDIST',
    'SITEDIST'
    ]

print(columns_to_prepare)

In [ ]:
# Here we use type again to confirm we have created a list
type(columns_to_prepare)

---

### 7 Selecting columns of a pandas DataFrame with a Python list

![Pandas select columns](images/03_subset_columns.svg)

*Image from: https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html (pandas documentation, BSD 3-Clause licence)*

We can now use the list to select the necessary DataFrame columns. Also, the list makes maintaining the code simpler. Any future changes in the required columns can be easily managed by just updating this list.

The line of code below `data_serving_layer = df[columns_to_prepare].copy()` filters the original DataFrame `data` and creates a new DataFrame `data_serving_layer` that only includes the columns specified in the `columns_to_prepare` list.

In [ ]:
# Filtering the dataset to keep only the requested columns
data_serving_layer = df[columns_to_prepare].copy()

Let's breakdown what that line of code above has done.

- `df[columns_to_prepare]`: This part of the code is an indexing operation. Instead of providing a single column name, a list of column names (`columns_to_prepare`) is passed to the [indexing operator (`[]`)](https://pandas.pydata.org/docs/user_guide/indexing.html#basics). This way, only the columns present in the list are selected, and a new DataFrame consisting of these selected columns is returned.

- `.copy()`: This function creates a copy of the filtered DataFrame. This step is important because without using `.copy()`, any changes made to `data_serving_layer` would also reflect in the original DataFrame `data`. We use `.copy()` to avoid this situation and ensure that further manipulations on `data_serving_layer` do not affect the original `data` DataFrame. This behaviour can also be [enabled globally](https://pandas.pydata.org/docs/user_guide/copy_on_write.html#how-to-enable-cow).

- Finally, this newly created DataFrame which only contains the selected columns is assigned to `data_serving_layer`.

In [ ]:
# Displaying the first few rows of the selected columns
data_serving_layer.head()

> 💻 Update the list `columns_to_prepare` in section 6 above with an additional column `SITEDIST_FLAG`. Then re-run the code after it to confirm you can add extra columns to the new DataFrame called `data_serving_layer`.

---

### 8 Select rows of a pandas DataFrame by value in a column

![Pandas select columns](images/03_subset_rows.svg)

*Image from: https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html (pandas documentation, BSD 3-Clause licence)*

As well as only being interested in certain columns, the data analyst is only interested in data related to ambulance arrivals. 

From the data dictionary, we know that ambulance arrivals are where the column `AEARRIVALMODE` is equal to 2 in our data.

To now filter the DataFrame by selecting certain *rows* we can use the line of in the code cell below. 

> 📗 Note in the code below, `data_serving_layer = data_serving_layer[data_serving_layer['AEARRIVALMODE'] == 2]`, the nested square brackets are used to create a "boolean mask" where `data_serving_layer['AEARRIVALMODE'] == 2` evaluates to `True` for each row, and this mask is then applied to `data_serving_layer` to filter out rows that do not meet this condition, effectively updating the DataFrame with only the rows where `AEARRIVALMODE` equals 2. https://www.programiz.com/python-programming/numpy/boolean-indexing

In [ ]:
data_serving_layer = data_serving_layer[data_serving_layer['AEARRIVALMODE'] == 2]

Next we use the `info()` method to print a concise summary of the DataFrame, including the data type (`Dtype`) column names, number of non-null values, and memory usage.

From the output, we can see that:

1. The DataFrame `data_serving_layer` has a total of 15 columns.
2. There are 7362 non-null entries in each column, meaning there are no missing values in the dataset. Non-null entries are basically the values present in a column of a DataFrame.
3. The two data types (`dtypes`) present in the dataset are integer (`int64`) and object (`object`). There are 13 columns of type `int64` and 2 columns of type `object`. Integer and object type columns can handle different kinds of operations. For instance, arithmetic operations can be performed on integer type columns but not on object type columns.

If there are non-numeric fields that need to be included in computations, they might convert those fields into numerical form. For instance, `ARRIVALDATE` is a string `object`. Later we will convert it into DateTime so we can more easily derive new fields from it.

In [ ]:
data_serving_layer.info()

---

### 9 Create new pandas DataFrame column from existing column

![Pandas derive columns](images/05_newcolumn_1.svg)

*Image from: https://pandas.pydata.org/docs/getting_started/intro_tutorials/05_add_columns.html (pandas documentation, BSD 3-Clause licence)*

The data analyst has requested you create new columns to assess the impact of relocation of A&E treatment sites. They ask you to model two scenarios: one where treatment sites are 25% closer and another where they are 25% farther from patients.

For closer sites, multiply `SITEDIST` by 0.75.
For farther sites, multiply `SITEDIST` by 1.25.

This will help in visualising how distance changes affect patient access to healthcare services on a dashboard.

In [ ]:
# Scenario 1: Closer Treatment Sites
data_serving_layer['SITEDIST_Closer'] = data_serving_layer['SITEDIST'] * 0.75

# Scenario 2: Farther Treatment Sites
data_serving_layer['SITEDIST_Farther'] = data_serving_layer['SITEDIST'] * 1.25

data_serving_layer[['SITEDIST','SITEDIST_Closer','SITEDIST_Farther']].head()

> 💻 The data analyst also wants the `SITEDIST` displayed in miles. The `SITEDIST` column currently shows the distance in kilometres between patient locations and A&E treatment sites. This conversion is essential for aligning the data with other datasets that use miles as the standard unit of distance, ensuring consistency in reporting. Create a new mile version of `SITEDIST` to change the distance measurements from kilometres to miles. The conversion factor from kilometres to miles is approximately 0.621371.

In [ ]:
# Create a new column that is SITEDIST converted from KM to miles


---

### 10 Change pandas DataFrame column data types

The data analyst also wants the dashboard to break down dates into individual components like year, month, day of the week, and hour, for uncovering trends and patterns in patient arrivals.

To achieve this we first convert the `ARRIVALDATE` column in the `data_serving_layer` DataFrame to a datetime format. 


The `pd.to_datetime` function from Pandas is used for this conversion, with the specified format `%Y-%m-%d` indicating the arrangement of year, month, and day in the date string.

In [ ]:
data_serving_layer['ARRIVALDATE'] = pd.to_datetime(data_serving_layer['ARRIVALDATE'], format='%Y-%m-%d')
data_serving_layer.info()

Next in the  code below we are creating a new column 'ARRIVAL_DATETIME' to combines the date and time into a single column.

The process involves converting both 'ARRIVALDATE' and 'ARRIVALTIME' columns to strings first. 

Then the 'ARRIVALTIME' string is padded with leading zeros to ensure a consistent HHMM format using `.str.pad(4, fillchar='0')`. 

These strings are then concatenated with a space between them to form a datetime string in the '%Y-%m-%d %H%M' format, which represents year-month-day hour-minute.

Finally the `pd.to_datetime` function is used to convert this combined string into a datetime object, which is then assigned to the new 'ARRIVAL_DATETIME' column.

In [ ]:
# Assuming ARRIVALTIME is in HHMM format
data_serving_layer['ARRIVAL_DATETIME'] = pd.to_datetime(
    data_serving_layer['ARRIVALDATE'].astype(str) + ' ' +
    data_serving_layer['ARRIVALTIME'].astype(str).str.pad(4, fillchar='0'),
    format='%Y-%m-%d %H%M'
)

> 💻 After successfully creating the `ARRIVAL_DATETIME` column, it's important to inspect the new column to ensure that the date and time have been combined and converted correctly. Use the `.info()` method on the dataframe `data_serving_layer` to confirm that the `ARRIVAL_DATETIME` column has the correct data type (datetime64[ns]).

In [ ]:
# Add .info() to the dataframe data_serving_layer


---

### 11 Extract parts of a pandas DataFrame date column

The data analyst has asked for year, month, and day of week to be extracted from the arrival datetime column created above. 

Below we are extracting specific date components from the `ARRIVAL_DATETIME` column in the `data_serving_layer` DataFrame and creating new columns for each component required. 

- `data_serving_layer['YEAR'] = data_serving_layer['ARRIVAL_DATETIME'].dt.year` extracts the year part from each datetime value in 'ARRIVAL_DATETIME' and assigns it to a new column named 'YEAR'.

- `data_serving_layer['MONTH'] = data_serving_layer['ARRIVAL_DATETIME'].dt.month` extracts the month component from the 'ARRIVAL_DATETIME' and stores it in a new column called 'MONTH'.

- `data_serving_layer['DAY_OF_WEEK'] = data_serving_layer['ARRIVAL_DATETIME'].dt.day_name()` extracts the day of the week from the 'ARRIVAL_DATETIME' and places it in a new column 'DAY_OF_WEEK'.

In [ ]:
data_serving_layer['YEAR'] = data_serving_layer['ARRIVAL_DATETIME'].dt.year
data_serving_layer['MONTH'] = data_serving_layer['ARRIVAL_DATETIME'].dt.month
data_serving_layer['DAY_OF_WEEK'] = data_serving_layer['ARRIVAL_DATETIME'].dt.day_name()
data_serving_layer['HOUR'] = data_serving_layer['ARRIVAL_DATETIME'].dt.hour

# Just display the new columns created and the original column they were derived from (ARRIVAL_DATETIME)
columns_to_display = ['ARRIVAL_DATETIME', 'YEAR', 'MONTH', 'DAY_OF_WEEK', 'HOUR']
data_serving_layer[columns_to_display].head()

> 💻 The data analyst has asked for the minutes to also be extracted and the column should be called `MINUTE`. Use the pandas documentation to work out the code you need and run it below https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.dt.minute.html

In [ ]:
# create a MINUTE column from ARRIVAL_DATETIME


---

### 12. Creating a new categorical column with If-Else inside a function

<img src="images/If-Then-Else-diagram.svg" width="500" alt="if else"/>

*Image from: https://en.wikipedia.org/wiki/Conditional_(computer_programming) (P. Kemp, CC0)*

In Python, functions are used for repeating tasks. They make code easier to manage. 

Also, if-else and elif (else-if) conditions control the execution flow based on certain conditions.

The function `categorise_duration` below demonstrates this. It uses if-else conditions to categorise durations into groups:

- Durations up to 60 minutes are 'Short'.
- Durations between 61 and 120 minutes are 'Medium'.
- Durations between 121 and 240 minutes are 'Long'.
- Durations over 240 minutes are 'Very Long'.

By using a function, the same logic can be applied across multiple DataFrame columns without repeating code and increasing the chance of making mistakes with repeated code. 


In [ ]:
def categorise_duration(duration):
    """
    Categorises a given duration into brackets.

    Parameters:
    - duration (int): The duration in minutes.

    Returns:
    - (str): A string label categorizing the duration.
    """
    if duration <= 60:
        return 'Short'     # Up to 1 hour
    elif duration <= 120:
        return 'Medium'    # 1 to 2 hours
    elif duration <= 240:
        return 'Long'      # 2 to 4 hours
    else:
        return 'Very Long' # More than 4 hours

We can now *apply* the function to a column such as the `TRETDUR` column in `data_serving_layer`.

The function `categorise_duration` is used to assign categories to a new column, `TRETDUR_CATEGORY`.

The `apply` method applies the `categorise_duration` function to each element in the `TRETDUR` column. 

Using [`apply`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html) is much more efficient and concise than manually iterating through every row in the column with a loop.

In [ ]:
# Example of applying the function to the 'TRETDUR' column
data_serving_layer['TRETDUR_CATEGORY'] = data_serving_layer['TRETDUR'].apply(categorise_duration)
data_serving_layer[['TRETDUR','TRETDUR_CATEGORY']].head()

> 💻 Update the function above to split the duration into 30 minute bands. Re-run the function to confirm your banding is correct.

---

### 13 Bulk apply a function across DataFrame columns with loop

<img src="images/For_loop_example.svg" width="200" alt="for loop"/>

*Image from: https://en.wikipedia.org/wiki/For_loop (CC BY-SA 4.0, https://commons.wikimedia.org/wiki/File:For_loop_example.svg)*

Loops are useful for running a block of code repeatedly. 

Below we use a `for` loop to iterate over a list of column names. For each column in the list, we apply the function created above to categorise the duration values to time columns one after the other. 

This loop saves us from having to write out similar lines of code for each individual column, making the code cleaner and simpler to adjust - for example, if we want to add or remove columns to process, we only need to edit the values in the the list.

In [ ]:
# List of duration columns to categorize
duration_columns = ['CONCLDUR', 'DEPDUR', 'INITDUR']

# Loop over the list and apply the categorise_duration function to each column
for col in duration_columns:
    category_col_name = col + '_CATEGORY'  # Create a new column name for categories
    data_serving_layer[category_col_name] = data_serving_layer[col].apply(categorise_duration)
data_serving_layer.head()

> 💻 In the code cell above, modify the list `duration_column` to include `TRETDUR` and run the cell above again to confirm the function has been applied to this column too.

---

### 14 Data Quality through automated EDA 

Following the core transformations the data analyst asked for, let's go on to explore the data to look for data quality issues such as unusual values. 

A good way to do this initially is through automated Exploratory Data Analysis (EDA). 

Creating plots manually helps us investigate what we want in fine-grained detail, which is essential for targeted analysis. However, for speed and efficiency, especially when dealing with large datasets or when we want a quick overview, we can make use of automated tools such as Sweetviz. 

Below we install and import Sweetviz, a Python library that generates fast high-level visualisations we can use for a quick data quality review.

In [ ]:
%pip install -q sweetviz

In [ ]:
import sweetviz as sv

# Analyze the DataFrame with a name
report = sv.analyze([data_serving_layer, "Data Serving Layer"])
# open_browser=False: the browser version cannot open a new browser window from Python
report.show_html('Sweetviz_Report.html', open_browser=False)

> 💻 Open the file `Sweetviz_Report.html` in the file browser on the left. It opens in a new tab; select **Trust HTML** at the top of that tab so the report's interactive charts work. Review the automatic plots. Are there any unusual values or distributions of values that you think might warrant a discussion with the data analyst about data quality?

> ℹ️ Sometimes, the `Sweetviz_Report.html` file may not be immediately visible in the file browser. If this happens, select the refresh button at the top of the file browser.

--- 
### 15 Write data to storage

![Pandas Read/Write](images/02_io_readwrite.svg)

*Image from: https://pandas.pydata.org/docs/getting_started/intro_tutorials/02_read_write.html (pandas documentation, BSD 3-Clause licence)*

So that we can share with the data analyst our initial work on the development data we can write the data to storage using one of the pandas writers.

Below we write out the data as a comma separated text file or CSV.

> 📗 Pandas `readers` and `writers` https://pandas.pydata.org/docs/user_guide/io.html#io

In [ ]:
data_serving_layer.to_csv('data_serving_layer.csv')

Writing Parquet files needs a Parquet engine. In the browser, pandas uses `fastparquet`, so install it first.

In [ ]:
%pip install -q fastparquet

> 💻 The data analyst has requested you share the data as a Parquet file. Using this guidance https://pandas.pydata.org/docs/user_guide/io.html#io, in the code cell below, now save the `data_serving_layer` DataFrame as a parquet file.

In [ ]:
# Write the DataFrame to Parquet format


> 🚀 After saving as a Parquet file, read the Parquet file into a different DataFrame name and check the it is equivalent to `data_serving_layer` using the Pandas compare function https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.compare.html

In [ ]:
# Read in the Parquet file and compare to the original DataFrame using pandas.compare()


--- 
### 16 Summary and your personal reflection

Let’s reflect on what we have learned through this module. 

#### Python and pandas skills

You ran this notebook in JupyterLite, a Python development environment that runs in your browser. Creating and using reliable and well-set-up development environments is a valuable skill in data engineering as it ensures consistent, replicable, and isolated development spaces. 

We walked through Python programming and the Pandas library focusing on data manipulation and analysis. This includes handling tasks with Pandas DataFrames like selection, filtering, and column creation. You will find these skills can help you prepare and structure data quickly for data engineering pipelines.

You have also learned to manipulate date and time data which can be a challenging area. This is important in data engineering given how often data contains timestamps and time-related information.

We also used automated visual data quality checks using an automated package, SweetViz. This is a fast way to check the quality of data by checking for missing and unusual values for example. Finding ways to assess data quality quickly and effectively is key to ensuring the accuracy and reliability of the data products you create.

#### Data Engineering Business Scenario

In the business scenario, you experienced the application of Python and Pandas skills on real development data that is safe to use. 

For example, carrying out exploratory analysis, selecting essential columns for a BI dashboard, and carrying out data transformations. This reflects real-world data engineering tasks with practical experience using Jupyter, Python, and pandas. These are not the only tools, languages and libraries that could have been used, but they are widely adopted and the basic principles are the same no matter which tools are used.

#### Personal reflection

> ❓ Can you apply the Python and Pandas skills learned in this module to your current or future projects?

> ❓ What types of data engineering problems might you now solve more efficiently with your new Python and Pandas skills?

> ❓ In what ways can you extend your knowledge of Python and Pandas to handle more complex data engineering tasks?

> ❓ How do you plan to keep your Python and Pandas skills updated with the rapidly changing technology landscape?

> ❓ What other tools, languages, or libraries are you interested in exploring to complement your Python and Pandas skills in data engineering?

> ❓ How can you use the experience of working with real development data in the business scenario for your future data engineering projects?